In [27]:
# Imports
import json
import concurrent.futures
import re
import os
from textwrap import dedent
from statistics import mean
from rich.console import Console
from dotenv import load_dotenv
from anthropic import Anthropic

# our custom functions packaged into python files
from my_chat_utils import (
    add_user_message,
    add_assistant_message,
    chat,
)
from report_utils import generate_prompt_evaluation_report
from prompt_evaluator import PromptEvaluator

In [28]:
# Client Initialization and helper functions
load_dotenv(override=True)

client = Anthropic()
model = "claude-haiku-4-5"
console = Console(force_jupyter=False)

In [29]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(client, model, max_concurrent_tasks=1)

In [ ]:
def generate_dataset(
    output_file_name: str = "dataset.json", force_generate: bool = False
):
    # generate dataset if output_file_name does NOT exist
    # OR if force_generate = True
    if force_generate or (not os.path.isfile(output_file_name)):
        dataset = evaluator.generate_dataset(
            # Describe the purpose or goal of the prompt you're trying to test
            task_description="Write a compact, concise 1 day meal plan for a single athlete",
            # Describe the different inputs that your prompt requires
            prompt_inputs_spec={
                "height": "Athlete's height in cm",
                "weight": "Athlete's weight in kg",
                "goal": "Goal of the athlete",
                "restrictions": "Dietary restrictions of the athlete",
            },
            # Where to write the generated dataset
            output_file=output_file_name,
            # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
            num_cases=3,
        )
    else:
        with open(output_file_name, "r") as f:
            dataset = json.load(f)

    return dataset


dataset = generate_dataset()

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [33]:
console.print(dataset)

[
    {
        'prompt_inputs': {
            'height': '178 cm',
            'weight': '68 kg',
            'goal': 'Marathon training with carbohydrate loading for endurance 
performance',
            'restrictions': 'None'
        },
        'solution_criteria': [
            'Meal plan contains exactly 4 meals (breakfast, lunch, snack, 
dinner) with estimated total calories between 3,000-3,500 kcal',
            'Carbohydrates comprise 55-65% of total macronutrients, with 
protein at 15-20% and fat at 20-25%',
            'Plan is presented in compact format (under 400 tokens) with meal 
timing and brief descriptions'
        ],
        'task_description': 'Write a compact, concise 1 day meal plan for a 
single athlete',
        'scenario': 'Testing with a high-endurance athlete (marathon runner) 
requiring high caloric intake and carbohydrate loading, with specific 
macronutrient ratios'
    },
    {
        'prompt_inputs': {
            'height': '180 cm',
            'weight':

In [34]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs, prompt):
    rendered_prompt = evaluator.render(prompt, prompt_inputs)

    messages = []
    add_user_message(messages, rendered_prompt)
    return chat(client, model, messages)

In [ ]:
# Prompt templates to evaluate.
# Placeholders like {height} are substituted per test case (from prompt_inputs)
# inside run_prompt via evaluator.render(prompt, prompt_inputs).

naive_prompt = """
What should this person eat?

- Height: {height}
- Weight: {weight}
- Goal: {goal}
- Dietary restrictions: {restrictions}
"""

# here you are being clearer about the meal plan
# - using direct words, such as "generate"
# - adding clear instructions - it should meet
# the athelete's dietary restriction
clear_and_direct_prompt = """
Generate a one-day meal plan for an athlete that 
meets their dietary restrictions.

- Height: {height}
- Weight: {weight}
- Goal: {goal}
- Dietary restrictions: {restrictions}
"""

# now you are adding guidelines over & above being
# clear & direct
clear_direct_and_specific_prompt = """
Generate a one-day meal plan for an athlete that 
meets their dietary restrictions.

- Height: {height}
- Weight: {weight}
- Goal: {goal}
- Dietary restrictions: {restrictions}

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned
"""

clear_direct_specific_with_xml = """
Generate a one-day meal plan for an athlete that 
meets their dietary restrictions.

<athlete_information> 
- Height: {height} 
- Weight: {weight} 
- Goal: {goal} 
- Dietary restrictions: {restrictions} 
</athlete_information>

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned
"""

engineered_prompt = """
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

<athlete_information> 
- Height: {height} 
- Weight: {weight} 
- Goal: {goal} 
- Dietary restrictions: {restrictions} 
</athlete_information>

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned

Here is an example with a sample input and an ideal output:
<sample_input>
height: 170
weight: 70
goal: Maintain fitness and improve cholesterol levels
restrictions: High cholesterol
</sample_input>
<ideal_output>
Here is a one-day meal plan for an athlete aiming to maintain fitness and improve cholesterol levels:

*   **Calorie Target:** Approximately 2500 calories
*   **Macronutrient Breakdown:** Protein (140g), Fat (70g), Carbs (340g)

**Meal Plan:**

*   **Breakfast (7:00 AM):** Oatmeal (80g dry weight) with berries (100g) and walnuts (15g). Skim milk (240g).
    *   Protein: 15g, Fat: 15g, Carbs: 60g
*   **Mid-Morning Snack (10:00 AM):** Apple (150g) with almond butter (30g).
    *   Protein: 7g, Fat: 18g, Carbs: 25g
*   **Lunch (1:00 PM):** Grilled chicken breast (120g) salad with mixed greens (150g), cucumber (50g), tomato (50g), and a light vinaigrette dressing (30g). Whole wheat bread (60g).
    *   Protein: 40g, Fat: 15g, Carbs: 70g
*   **Afternoon Snack (4:00 PM):** Greek yogurt (170g, non-fat) with a banana (120g).
    *   Protein: 20g, Fat: 0g, Carbs: 40g
*   **Dinner (7:00 PM):** Baked salmon (140g) with steamed broccoli (200g) and quinoa (75g dry weight).
    *   Protein: 40g, Fat: 20g, Carbs: 80g
*   **Evening Snack (9:00 PM):** Small handful of almonds (20g).
    *   Protein: 8g, Fat: 12g, Carbs: 15g

This meal plan prioritizes lean protein sources, whole grains, fruits, and vegetables, while limiting saturated and trans fats to support healthy cholesterol levels.
</ideal_output>
This example meal plan is well-structured, provides detailed information on food choices and quantities, and aligns with the athlete's goals and restrictions.
"""

In [36]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    prompt=naive_prompt,
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 2.6666666666666665


In [37]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    prompt=clear_and_direct_prompt,
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 5.666666666666667


In [39]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    prompt=clear_direct_and_specific_prompt,
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 6.333333333333333


In [42]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    prompt=clear_direct_specific_with_xml,
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 6.333333333333333
